## Lower Path (Global Trends Analyzer)

**Core Objective:** Identify overarching macroeconomic shifts and broad market narratives by semantically clustering related news topics ("What is the bigger picture?").

**Pipeline Steps:**
1. **Vectorization:** Converts the textual content (title and description) of each topic into high-dimensional numerical embeddings using an LLM embedding model (e.g., `text-embedding-3-small`), capturing their semantic meaning.
2. **Clustering:** Groups semantically similar topics together using the HDBSCAN algorithm. This process isolates distinct macro trends and effectively filters out irrelevant "noise" (isolated topics not forming a trend).
3. **Severity Calculation:** Computes the signal strength (`severity`) of each identified cluster, primarily based on its size (number of topics involved), ensuring only significant trends are processed.
4. **Trend Generation:** Synthesizes the topics within each valid cluster via LLM to output a consolidated global narrative, determining its broad market direction (`up`, `down`, `neutral`) and a concise summary.

**Output:** `GLOBAL_TRENDS` DataFrame containing overarching macroeconomic narratives and their general market direction.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# 1. Import the analyzer AND the central Config
from nlp_predictor.config import Config
from nlp_predictor.global_trends import LowerPathAnalyzer

# 2. Resolve the project root from the notebook working directory
notebook_dir = Path.cwd().resolve()
BASE_DIR = notebook_dir.parent

# 3. Ensure the 'src' directory is on sys.path so the local package can be imported
src_dir = BASE_DIR / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print("====== LowerPathAnalyzer Module Test ======")

try:
    lower_path_analyzer = LowerPathAnalyzer(Config["INSTRUMENT_PATH"])
    df_topics = pd.read_csv(Config["TOPICS_PATH"])

    # FIX: Changed from process_lower_path() to the abstract process_data()
    result = lower_path_analyzer.process_data(df_topics)
    print("====== Processing Lower Path Completed ======")

    print("====== Saving Results To File ======")
    saving_result = LowerPathAnalyzer.save_trends(
        result, Config["GLOBAL_TRENDS_SAVE_PATH_PARQUET"]
    )
    if saving_result:
        print("====== Saving Completed ======")
    else:
        print("====== Saving Failed ======")

    # For debugging: save the result to a CSV as well
    result.to_csv(Config["GLOBAL_TRENDS_SAVE_PATH_CSV"], index=False)
    print(
        "[DEBUG] Also saved results to CSV for quick inspection at "
        + f"{Config['GLOBAL_TRENDS_SAVE_PATH_CSV']}"
    )

except Exception as e:
    print(f"[FATAL ERROR] An error occurred during the LowerPathAnalyzer test: {e}")